# 12-amaliyot — Transformer bilan matn umumlashtirish (m12)

**Mavzu:** Transformer arxitekturasi yordamida matnlardan qisqa xulosalar chiqarish.
**Kuni:** 13-kun (Day 13) · **Juft ma'ruza:** [L12 — Transformer](../lectures/d12_transformer.pdf)
**Quriladigan kapstone moduli:** `m12 TransformerSummarizer` (haqiqiy modul — `consumed_by`: m15 agent, Day 16 pipeline).

Bugun L12 da o'rgangan **self-attention**, **multi-head attention** va **positional encoding** ni
amalda quramiz va kichik maqola-xulosa korpusida Transformer enkoder-dekoderni o'qitamiz.

**Bugungi maqsadlar (L12 [C] dan):**
1. Self-attention va positional encoding ni numpy'da implementatsiya qilamiz.
2. Mini Transformer enkoder-dekoder bilan xulosa generatsiya qilamiz.
3. Natijani **ROUGE-1** bilan baholaymiz (P=1.0, R=0.6, F1=0.75 — L12 [I4]).

**Vaqt taqsimoti:** Muhit 5 daq · Yaxlit natija 8 daq · Periferiya 17 daq · Yadro 35 daq · Ulash 10 daq · Yakun 5 daq.

> ⚠️ **Korpus litsenziyasi:** Wikipedia uz lead-paragraph juftlari (CC-BY-SA 3.0; manba: Wikipedia
> hissadorlari). Offline rejimda kichik original uz maqola-xulosa korpusi ishlatiladi.

## 1. Muhit tekshiruvi

Kutubxonalar, urug' (seed) va `OFFLINE_FALLBACK` bayrog'ini sozlaymiz. `torch` yoki `sentencepiece`
bo'lmasa ham notebook uchdan-uchgacha ishlaydi (zaxira yo'llar).

In [ ]:
import os, sys, random, warnings
warnings.filterwarnings("ignore")
import numpy as np

# Kaggle GPU bo'lsa ishlatiladi; mahalliy CPU ham yetarli (kichik korpus)
OFFLINE_FALLBACK = True          # internetdan yuklab bo'lmasa, bundlangan korpus

random.seed(42); np.random.seed(42)

try:
    import torch
    torch.manual_seed(42)
    HAS_TORCH = True
except ImportError:
    HAS_TORCH = False

try:
    import sentencepiece          # noqa: F401
    HAS_SP = True
except ImportError:
    HAS_SP = False

print("numpy:", np.__version__, "| torch:", (torch.__version__ if HAS_TORCH else "yo'q"),
      "| sentencepiece:", ("bor" if HAS_SP else "yo'q"))
print("OFFLINE_FALLBACK =", OFFLINE_FALLBACK)

# kapstone modul yo'lini topamiz (mahalliy yoki Kaggle)
for _cand in ["../../capstone/modules", "/kaggle/working/capstone/modules", "capstone/modules"]:
    if os.path.isdir(_cand):
        sys.path.insert(0, _cand); break

CKPT = "d13_checkpoints"
CORPUS_PATH = os.path.join(CKPT, "uz_wiki_summ_mini.txt")

## 2. Yaxlit natija (avval manzil)

Avval tugallangan natijani ko'ramiz: kichik korpusda Transformer'ni o'qitamiz va bir maqola uchun
avtomatik xulosa olamiz. Tafsilotlarni keyin ochamiz.

In [ ]:
from m12_transformer_summarizer import TransformerSummarizer

def load_pairs(path):
    """Maqola<TAB>xulosa juftlarini o'qiydi. Wikipedia uz (onlayn) yoki offline mini korpus."""
    pairs = []
    # Kaggle (onlayn): bu yerda datasets.load_dataset(...) bo'lardi; offline bundlangan faylga tushamiz
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.rstrip("\n")
            if "\t" in line:
                maqola, xulosa = line.split("\t", 1)
                pairs.append((maqola.strip(), xulosa.strip()))
    return pairs

pairs = load_pairs(CORPUS_PATH)
maqolalar = [m for m, x in pairs]
xulosalar = [x for m, x in pairs]
print("Juftlar soni:", len(pairs))

demo = TransformerSummarizer()
demo.train(maqolalar, xulosalar, epochs=20, d_model=64, nhead=4)   # to'liq: GPU, 10+ epoch, d_model=128
xulosa = demo.summarize(maqolalar[0], max_length=30)
print("\nMaqola :", maqolalar[0])
print("Xulosa :", xulosa)
print("ROUGE-1:", {k: round(v, 3) for k, v in demo.rouge1([xulosalar[0]], [xulosa]).items()})

## 3. Tayyor kod bloki — periferiya (PRIMM)

Bu yerdagi kod **to'liq berilgan** (bugungi o'rganish maqsadi emas, ammo to'liqlik uchun zarur):
korpus yuklash, **tokenizatsiya** (sentencepiece BPE bo'lsa undan, aks holda so'z-darajali) va
enkoder-dekoder uchun ketma-ketlikka aylantirish.

### Bashorat qiling
Quyidagi katak bitta maqolani tokenlarga ajratadi. Sizningcha, `"Toshkent O'zbekistonning poytaxti."`
nechta token beradi? Apostrofli `O'zbekistonning` bitta token bo'lib qoladimi?

In [ ]:
def tokenize(text):
    """sentencepiece bo'lsa BPE; aks holda sodda so'z-darajali tokenizatsiya (HAS_SP bayrog'i)."""
    if HAS_SP:
        # Kaggle yo'li: oldindan o'qitilgan SentencePiece modeli (vocab=5000) ishlatilardi
        return text.split()       # (demo uchun soddalashtirildi)
    return text.split()

namuna = maqolalar[0]
toklar = tokenize(namuna)
print("Maqola   :", namuna)
print("Tokenlar :", toklar)
print("Token soni:", len(toklar))

# Enkoder-dekoder uchun statistika (scaffold — to'liq beriladi)
uzunliklar = [len(tokenize(m)) for m in maqolalar]
print("\nMaqola uzunligi (token): min=%d, o'rtacha=%.1f, max=%d" % (
      min(uzunliklar), sum(uzunliklar) / len(uzunliklar), max(uzunliklar)))

### Tekshiring
1. `O'zbekistonning` so'zi nechta tokenga bo'lindi? (Sodda tokenizatorda bittaga.)
2. Haqiqiy Transformer'da nega **BPE** (sub-word) ishlatiladi? (O'zbek qo'shimchalari: `kitob-im-ni`.)
3. `max` token soni `positional encoding` uzunligidan oshmasligi kerak — nega?

### O'zgartiring
`namuna` o'rniga `maqolalar[3]` ni qo'yib, boshqa maqolaning tokenlarini ko'ring. Yoki o'z
matningizni `tokenize("...")` ga bering.

### ✅ Checkpoint A
Quyidagi katak korpusni qaytadan yuklaydi — agar oldingi kataklar ishlamagan bo'lsa, shu yerdan
davom etish mumkin (ishchi holatga qaytamiz).

In [ ]:
if OFFLINE_FALLBACK or "pairs" not in dir():
    pairs = load_pairs(CORPUS_PATH)
    maqolalar = [m for m, x in pairs]
    xulosalar = [x for m, x in pairs]
print("Checkpoint A: %d juft tayyor." % len(pairs))

## 4. Asosiy mavzu — so'nuvchi tayanch

Bugungi yadro: **self-attention**, **positional encoding** va **multi-head attention** ni
numpy'da quramiz (L12 nazariyasi). Avval namunani ko'ramiz, so'ng birgalikda yozamiz, oxirida
mustaqil qo'llaymiz.

### 4A. Namuna (men qilaman): ROUGE-1 va attention softmax

L12 [I4] dagi **ROUGE-1** ni qo'lda tekshiramiz (qulflangan natija) va [I1] dagi attention
softmax'ini eslaymiz.

In [ ]:
# --- Qulflangan natija: ROUGE-1 (L12 [I4]) ---
sum0 = TransformerSummarizer()
r = sum0.rouge1(["nlp juda qiziq va foydali"], ["nlp juda foydali"])
print("ROUGE-1:", {k: round(v, 3) for k, v in r.items()})
assert abs(r["precision"] - 1.000) < 1e-3
assert abs(r["recall"]    - 0.600) < 1e-3
assert abs(r["f1"]        - 0.750) < 1e-3   # Ma'ruza L12 [I4]-slayd bilan solishtiring
print("To'g'ri! ROUGE-1 F1 = 0.750 (L12 [I4] bilan mos).")

# --- Attention softmax (L12 [I1]) qo'lda izoh ---
e = np.array([2.0, 1.0, 3.0])               # masshtablangan energiyalar (q.k / sqrt(d_k))
alpha = np.exp(e) / np.exp(e).sum()
print("alpha = softmax([2,1,3]) =", np.round(alpha, 3))   # [0.245 0.090 0.665]

### 4B. Birgalikda (biz qilamiz): scaled dot-product attention

`Attention(Q, K, V) = softmax(QKᵀ / √dₖ) · V`. Quyidagi funksiyani to'ldiring.

In [ ]:
import numpy as np

def scaled_dot_product_attention(Q, K, V):
    """Attention(Q,K,V) = softmax(QK^T / sqrt(d_k)) V. Qaytaradi: (kontekst, alpha)."""
    d_k = K.shape[-1]
    # === SIZNING KODINGIZ (taxminan 5 qator) ===
    # 1) scores = Q @ K.T / sqrt(d_k)
    # 2) barqarorlik uchun har qatordan max ni ayiring, so'ng exp
    # 3) alpha = qatorlar bo'yicha normallangan (yig'indi = 1)
    # 4) context = alpha @ V ; (context, alpha) ni qaytaring
    raise NotImplementedError("scaled_dot_product_attention ni to'ldiring")

In [ ]:
Q = np.array([[4., 2., 6., 0.]])                          # bitta so'rov (q . k = [4,2,6])
K = np.array([[1, 0, 0, 0], [0, 1, 0, 0], [0, 0, 1, 0]], float)  # 3 kalit, d_k = 4
V = np.array([[1, 0], [0, 1], [1, 1]], float)             # 3 qiymat
context, alpha = scaled_dot_product_attention(Q, K, V)
print("alpha   :", np.round(alpha[0], 3))
print("kontekst:", np.round(context[0], 3))
assert np.allclose(alpha[0], [0.245, 0.090, 0.665], atol=1e-3)  # Ma'ruza L12 [I1]-slayd bilan solishtiring
assert np.allclose(context[0], [0.910, 0.755], atol=1e-3)
print("To'g'ri! Self-attention vaznlari L12 [I1] bilan mos.")

### 4C. Birgalikda: positional encoding

Self-attention so'z **tartibini** ko'rmaydi — sinusoidal PE qo'shamiz:
`PE(pos,2i)=sin(pos/10000^(2i/d))`, `PE(pos,2i+1)=cos(...)`.

In [ ]:
def positional_encoding(n_pos, d):
    """Sinusoidal PE: juft o'lcham -> sin, toq o'lcham -> cos."""
    pe = np.zeros((n_pos, d))
    pos = np.arange(n_pos)[:, None]
    i = np.arange(d)[None, :]
    angle = pos / np.power(10000.0, (2 * (i // 2)) / d)
    # === SIZNING KODINGIZ (taxminan 2 qator) ===
    # juft ustunlarga sin(angle), toq ustunlarga cos(angle) ni yozing ([0::2], [1::2])
    raise NotImplementedError("sin/cos qatorlarini to'ldiring")
    return pe

In [ ]:
pe = positional_encoding(2, 4)
print("PE(0):", np.round(pe[0], 3))
print("PE(1):", np.round(pe[1], 3))
assert np.allclose(pe[0], [0, 1, 0, 1], atol=1e-6)   # Ma'ruza L12 [I2]-slayd bilan solishtiring
print("To'g'ri! PE(0) = [0,1,0,1].")

### 4D. Birgalikda: multi-head attention

Bir nechta head — har biri `d/nhead` o'lchamli alohida attention. Natijalar birlashtiriladi.

In [ ]:
def multi_head_attention(X, Wq, Wk, Wv, Wo, nhead):
    """X:(n,d). nhead ta head, har biri d/nhead o'lchamli scaled dot-product."""
    n, d = X.shape
    dk = d // nhead
    Q, K, V = X @ Wq, X @ Wk, X @ Wv
    heads = []
    # === SIZNING KODINGIZ (taxminan 4 qator) ===
    # har head uchun Q/K/V ning [h*dk:(h+1)*dk] bo'lagini scaled_dot_product_attention ga bering,
    # chiqishlarni heads ga qo'shing, so'ng axis=1 bo'yicha concat qiling
    raise NotImplementedError("head'lar siklini to'ldiring")
    return concat @ Wo

In [ ]:
rng = np.random.RandomState(0)
d, nhead, n = 8, 2, 5
X = rng.randn(n, d)
Wq, Wk, Wv, Wo = [rng.randn(d, d) for _ in range(4)]
Y = multi_head_attention(X, Wq, Wk, Wv, Wo, nhead)
print("kirish shakli:", X.shape, " chiqish shakli:", Y.shape)
assert Y.shape == (n, d)
assert np.isfinite(Y).all()
print("To'g'ri! Multi-head chiqishi kirish o'lchamini (n, d) saqladi.")

### 4E. Mustaqil (siz qilasiz): o'z korpusingizda umumlashtiring

`m12 TransformerSummarizer` ni korpusda o'qitib, bir maqola uchun xulosa oling va ROUGE-1 bilan
baholang. (Kichik data — xulosa va ROUGE demo-sifatli, bu halol.)

In [ ]:
sum2 = TransformerSummarizer()
sum2.train(maqolalar, xulosalar, epochs=20, d_model=64, nhead=4)

matn = maqolalar[1]
xulosa2 = sum2.summarize(matn, max_length=30)
res = sum2.rouge1([xulosalar[1]], [xulosa2])
print("Maqola :", matn)
print("Xulosa :", xulosa2)
print("ROUGE-1:", {k: round(v, 3) for k, v in res.items()})

In [ ]:
# strukturaviy tekshiruv (aniq matn emas — kichik data, demo-sifatli)
assert isinstance(xulosa2, str)
assert set(res.keys()) == {"precision", "recall", "f1"}
assert all(0.0 <= res[k] <= 1.0 for k in res)
print("To'g'ri! summarize -> str; rouge1 -> 3 kalitli dict, har biri [0,1].")

## 5. Loyihaga ulash — `m12 TransformerSummarizer`

Bugungi kod `capstone/modules/m12_transformer_summarizer.py` ga yig'ilgan (interfeys
`contracts.py` dan). U **haqiqiy modul**: `save`/`load` bor va keyingi kunlarda m15 agent
(`summarize_text`) hamda Day 16 pipeline ishlatadi.

In [ ]:
import tempfile
# shartnoma mosligi
api = TransformerSummarizer()
for m in ("train", "summarize", "rouge1", "save", "load"):
    assert hasattr(api, m), "yetishmayotgan metod: " + m

# save -> load -> summarize ishlashini tekshiramiz (consumed_by [15,16])
api.train(maqolalar, xulosalar, epochs=10, d_model=64, nhead=4)
_p = os.path.join(tempfile.gettempdir(), "m12_test.pkl")
api.save(_p)
api2 = TransformerSummarizer()
api2.load(_p)
qayta = api2.summarize(maqolalar[0], max_length=30)
print("Yuklangan modeldan xulosa:", qayta)
assert isinstance(qayta, str)
os.remove(_p)
print("To'g'ri! m12 shartnomaga mos; save/load ishlaydi.")

**Git (o'z repozitoriyangizda):**
```bash
git add capstone/modules/m12_transformer_summarizer.py
git commit -m "P12: m12 TransformerSummarizer qo'shildi"
git push
```

## 6. Tadqiqot savoli + yakun

**Tajriba:** `m12` ning ikki yo'li bor — **abstraktiv** (torch Transformer, yangi so'zlar yaratadi)
va **ekstraktiv** (torch'siz, mavjud gaplardan tanlaydi). Quyida ekstraktiv yo'lni majburan
ko'ramiz va abstraktiv natija bilan solishtiramiz.

In [ ]:
import m12_transformer_summarizer as M12
saved = M12.HAS_TORCH
try:
    M12.HAS_TORCH = False                       # ekstraktiv yo'lni majburlash
    ext = M12.TransformerSummarizer()
    ext.train(maqolalar, xulosalar, epochs=1)   # ekstraktiv: o'qitish yengil
    print("Ekstraktiv xulosa:", ext.summarize(maqolalar[0], max_length=12))
finally:
    M12.HAS_TORCH = saved
print("\nAbstraktiv (torch) xulosa:", demo.summarize(maqolalar[0], max_length=30))

**O'ylab ko'ring:** ekstraktiv usul har doim grammatik to'g'ri gap beradi, lekin yangi
ifoda yarata olmaydi; abstraktiv usul moslashuvchan, ammo kichik datada xato qiladi. O'zbek
qo'shimchalari (`kitob-im-ni`) qaysi usulga ko'proq qiyinchilik tug'diradi — nega?

---
**Chiqish chiptasi:** Bugun eng tushunarsiz qolgan narsa nima? (Bir jumla.)